In [35]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [36]:
!pip install -U \
  fastapi \
  uvicorn \
  pyngrok \
  "transformers>=4.41.0" \
  "peft>=0.11.1" \
  "bitsandbytes>=0.43.1" \
  "accelerate>=0.30.0" \
  torch \
  pydantic


In [37]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
!ls /content/drive/MyDrive/healthlens_medchat_lora/final_release


adapter_config.json	   README.md		    tokenizer.json
adapter_model.safetensors  special_tokens_map.json  tokenizer.model
chat_template.jinja	   tokenizer_config.json


In [39]:
%%writefile app.py
# =========================
# HealthLens FastAPI (Colab)
# =========================
import os
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
ADAPTER_DIR = "/content/drive/MyDrive/healthlens_medchat_lora/final_release"
MAX_SEQ_LEN = 2048
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def is_non_medical(text: str) -> bool:
    triggers = ["hi", "hello", "hey", "how are you"]
    t = text.lower().strip()
    return any(t == x or t.startswith(x) for x in triggers)

SYSTEM_PROMPT = (
    "You are HealthLens, a careful and conservative medical assistant.\n\n"
    "Rules:\n"
    "1) Do NOT provide a medical diagnosis.\n"
    "2) Do NOT recommend prescription medications unless clearly appropriate.\n"
    "3) Do NOT invent symptoms.\n"
    "4) Address only what the user mentions.\n"
    "5) If non-medical, respond briefly.\n"
    "6) Ask clarifying questions if needed.\n"
    "7) Advise emergency care if appropriate.\n"
    "8) Say when uncertain.\n"
    "9) Prefer safety over speculation.\n"
)

app = FastAPI(title="HealthLens API", version="1.0.0")

class ChatRequest(BaseModel):
    message: str
    max_new_tokens: int = 256
    temperature: float = 0.7
    top_p: float = 0.9

class ChatResponse(BaseModel):
    response: str

print("🔹 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

print("🔹 Loading base model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("🔹 Attaching LoRA...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("✅ HealthLens model ready")

def format_chat(system, user):
    return f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"

@app.post("/chat", response_model=ChatResponse)
@torch.no_grad()
def chat(req: ChatRequest):
    if is_non_medical(req.message):
        return ChatResponse(response="Hello! How can I help you today?")

    prompt = format_chat(SYSTEM_PROMPT, req.message)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=req.max_new_tokens,
        temperature=req.temperature,
        top_p=req.top_p,
        do_sample=req.temperature > 0,
        eos_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    decoded = decoded.split("<|assistant|>")[-1].strip()
    return ChatResponse(response=decoded)

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}


Overwriting app.py


In [40]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8004 > server.log 2>&1 &


In [41]:
!sleep 3
!tail -n 20 server.log


In [42]:
!ps aux | grep uvicorn | grep -v grep


root       16116 71.3  3.8 4623652 514736 ?      Rl   03:27   0:02 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 8004


In [43]:
!grep -n "Uvicorn" server.log

In [44]:
from pyngrok import ngrok

ngrok.set_auth_token("35zjZUE5cNl3MnC7LSe8D4rlVBJ_357K5pTMweBCpAutM2sxh")

public_url = ngrok.connect(8004)
print(public_url)


NgrokTunnel: "https://booker-unsporting-colene.ngrok-free.dev" -> "http://localhost:8004"


In [45]:
from pyngrok import ngrok
print(ngrok.get_tunnels())


[<NgrokTunnel: "https://booker-unsporting-colene.ngrok-free.dev" -> "http://localhost:8004">]


In [46]:
!ps aux | grep uvicorn | grep -v grep


root       16116 68.0  3.9 4640676 529764 ?      Rl   03:27   0:02 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 8004


In [1]:
!curl http://127.0.0.1:8004/health

curl: (7) Failed to connect to 127.0.0.1 port 8004 after 0 ms: Connection refused


In [48]:
# !pkill -f uvicorn

#kill

In [49]:
# ngrok.kill()
# !pkill -f uvicorn
# !ps aux | grep uvicorn | grep -v grep